# AUA Dataset v1 — аудит готовності набору даних

**Для студентів і керівників НДР | версія 1.0**

Цей ноутбук перевіряє **наявний каталог `AUA`**, але **не змінює і не видаляє жодного файла**. Результати формуються в окремій папці `results/AUA_audit/deep_audit`.

**Що перевіримо:**
1. Усі каталоги, кількість і розміри файлів.
2. **Аудіо:** справжні години, власне / зовнішнє / синтетичне, доступні ID мовців.
3. **Відео:** кількість власних master-файлів, зовнішніх джерел і маніпуляцій; FPS, роздільність, тривалість.
4. **Тексти:** кількість *кандидатів на унікальні кейси* за `case_id` / файлами, наявність двох мов.
5. **Splits:** точні дублікати за SHA-256, імовірні повтори мовців, метадані походження та зв’язок master → variant.
6. **KPI:** що вже підтверджено, що не виконано, а що потребує ручної перевірки.

**Межі автоматичної перевірки:** назви файлів самі по собі не доводять унікальності мовця, належності до діалекту, семантичної незалежності текстових кейсів, чинності ліцензії чи згоди. Такі пункти отримують `CHECK`, а не безпідставний `PASS`.

**Запуск:** відкрийте файл у JupyterLab та оберіть **Run → Run All Cells**. Якщо каталог не знайдено, задайте `DATA_ROOT` у першому кодовому блоці.

## 1. Налаштування і пошук AUA

Зазвичай змінювати нічого не потрібно. Якщо ноутбук відкритий у `/home/jovyan/work/.../AUA`, він знайде каталог автоматично. За потреби вкажіть абсолютний шлях у `DATA_ROOT` (наприклад `/home/jovyan/work/Melnyk_dus/AUA`).

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os, json, re, csv, wave, hashlib, subprocess, shutil, time
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Можна вказати шлях вручну (або залишити None).
DATA_ROOT = None
# Для тривалої операції SHA-256 в усіх підозрілих парах між split — True.
CHECK_EXACT_SPLIT_DUPLICATES = True
# Не змінювати файли вихідного набору.

EXPECTED = ["00_sources_and_licenses", "01_audio", "02_video", "03_text", "04_docs", "05_splits"]

def find_aua():
    explicit = DATA_ROOT or os.getenv("AUA_DATA_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"Каталог не існує: {candidate}")
        return candidate
    cwd = Path.cwd().resolve()
    bases = list(dict.fromkeys([cwd, *cwd.parents, Path.home(), Path.home()/"work",
                                 Path("/home/jovyan/work"), Path("/home/jovyan/work/Melnyk_dus")]))
    candidates = []
    for base in bases:
        candidates.extend([base, base/"AUA"])
        if base.exists() and base.is_dir() and str(base) not in ("/", "/home"):
            # Тільки безпосередні дочірні папки, не повний рекурсивний пошук.
            try:
                candidates.extend(p for p in base.iterdir() if p.is_dir() and p.name.lower() == "aua")
            except OSError:
                pass
    ranked = []
    for p in dict.fromkeys(candidates):
        if p.is_dir():
            score = sum((p / name).is_dir() for name in EXPECTED)
            if score >= 3:
                ranked.append((score, len(str(p)), p.resolve()))
    if not ranked:
        raise FileNotFoundError("Не знайдено AUA. Вкажіть DATA_ROOT = '/абсолютний/шлях/AUA'.")
    return sorted(ranked, key=lambda x:(-x[0], x[1]))[0][2]

DATA_ROOT = find_aua()
OUT = Path(os.getenv("AUA_AUDIT_OUT", str(DATA_ROOT.parent / "results" / "AUA_audit" / "deep_audit"))).expanduser().resolve()
# Виведення винесене за межі набору, щоб аудит не рахував свої ж результати.
if DATA_ROOT == OUT or DATA_ROOT in OUT.parents:
    OUT = DATA_ROOT.parent / "AUA_deep_audit_results"
OUT.mkdir(parents=True, exist_ok=True)
FFPROBE = shutil.which("ffprobe")
print("Набір:       ", DATA_ROOT)
print("Звіти:       ", OUT)
print("ffprobe:     ", FFPROBE or "НЕ ЗНАЙДЕНО: WAV перевіримо, для MP3/MP4 потрібен ffmpeg/ffprobe")
print("Каталоги:")
for folder in EXPECTED:
    print(f"  {folder:<28} {'OK' if (DATA_ROOT/folder).is_dir() else 'НЕМАЄ'}")


## 2. Інвентаризація всіх файлів

**Один фізичний файл = один запис маніфесту.** Файли `05_splits` не додаються повторно до обсягів оригінальних аудіо й відео.

In [ ]:
AUDIO_EXT = {".wav", ".mp3", ".m4a", ".aac", ".ogg", ".flac", ".opus"}
VIDEO_EXT = {".mp4", ".webm", ".mov", ".mkv", ".avi", ".m4v"}
STRUCTURED_EXT = {".csv", ".tsv", ".json", ".jsonl", ".parquet", ".xlsx"}
SERVICE_EXT = {".pyc", ".pack", ".idx", ".rev", ".sample"}

def relpath(p):
    return Path(p).relative_to(DATA_ROOT).as_posix()

rows = []
for current, dirs, files in os.walk(DATA_ROOT, followlinks=False):
    dirs[:] = [d for d in dirs if not (Path(current)/d).is_symlink()]
    for fn in files:
        p = Path(current)/fn
        if p.is_symlink():
            continue
        try:
            r = p.relative_to(DATA_ROOT)
            rows.append({"relative_path": r.as_posix(), "section": r.parts[0],
                         "extension": p.suffix.lower(), "size_bytes": p.stat().st_size,
                         "filename": fn, "path": str(p)})
        except OSError:
            continue
inventory = pd.DataFrame(rows)
if inventory.empty:
    raise RuntimeError("У DATA_ROOT немає файлів. Перевірте шлях.")
inventory.drop(columns=["path"]).to_csv(OUT/"01_file_inventory.csv", index=False)
folder_table = (inventory.groupby("section").agg(files=("filename","size"),
                 gib=("size_bytes", lambda x: x.sum()/1024**3)).reset_index())
print(f"Усього: {len(inventory):,} файлів; {inventory.size_bytes.sum()/1024**3:.3f} GiB")
display(folder_table.round(3))
print("Формати (ТОП 12):")
display(inventory.extension.replace("", "[без розширення]").value_counts().head(12).to_frame("файлів"))


## 3. Допоміжні функції та огляд метаданих

Для кожної таблиці з метаданими запишемо її поля й кількість рядків. **Поки не встановлено зв'язок ID з конкретним файлом, чисельність мовців і джерел не вважатиметься підтвердженою.**

In [ ]:
def read_table(p, max_mib=30):
    """Обмежене читання структурованих даних. Повертає DataFrame або None."""
    try:
        p = Path(p)
        if p.stat().st_size > max_mib*1024**2:
            return None
        ext = p.suffix.lower()
        if ext == ".csv": return pd.read_csv(p, low_memory=False)
        if ext == ".tsv": return pd.read_csv(p, sep="\t", low_memory=False)
        if ext == ".xlsx": return pd.read_excel(p)
        if ext == ".parquet": return pd.read_parquet(p)
        if ext == ".jsonl": return pd.read_json(p, lines=True)
        if ext == ".json":
            with p.open("r", encoding="utf-8") as f: obj = json.load(f)
            if isinstance(obj, list) and (not obj or isinstance(obj[0], dict)):
                return pd.DataFrame(obj)
            if isinstance(obj, dict):
                for k in ("cases", "items", "records", "data", "examples", "results"):
                    v = obj.get(k)
                    if isinstance(v, list) and (not v or isinstance(v[0], dict)):
                        return pd.DataFrame(v)
                return pd.DataFrame([obj])
    except Exception:
        pass
    return None

def parse_speaker_id(name):
    m = re.search(r"(?<![A-Za-z0-9])(S\d{2,5}|SPK[_-]?\d{1,5}|SPEAKER[_-]?\d{1,5})(?![A-Za-z0-9])", str(name), re.I)
    return m.group(1).upper().replace("-", "_") if m else None

def fmt_h(hours):
    return "невідомо" if pd.isna(hours) else f"{hours:.3f} год"

meta_rows = []
meta_roots = [DATA_ROOT/"01_audio", DATA_ROOT/"02_video", DATA_ROOT/"03_text", DATA_ROOT/"04_docs"]
for root in meta_roots:
    if not root.exists(): continue
    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in STRUCTURED_EXT: continue
        df = read_table(p)
        meta_rows.append({"file": relpath(p), "rows": len(df) if df is not None else np.nan,
                          "columns": " | ".join(map(str, df.columns)) if df is not None else "не прочитано / надто великий",
                          "speaker_field": any(str(c).lower() in {"speaker_id","speaker","client_id","person_id","identity_id"} for c in df.columns) if df is not None else False})
metadata_inventory = pd.DataFrame(meta_rows, columns=["file", "rows", "columns", "speaker_field"])
metadata_inventory.to_csv(OUT/"02_metadata_inventory.csv", index=False)
print(f"Метаданих перевірено: {len(metadata_inventory)} файлів")
display(metadata_inventory.head(20))


## 4. Аудіо: реальна тривалість і склад

Для WAV тривалість визначається без декодування файлу. Для MP3, M4A, OGG тощо потрібен `ffprobe`. Тривалості з `own_master` показуються **інформаційно**, але не додаються до `own_processed` у KPI.

In [ ]:
def probe_media(p):
    info = {"duration_sec":np.nan, "fps":np.nan, "width":np.nan, "height":np.nan,
            "audio_codec":None, "video_codec":None, "sample_rate":np.nan, "channels":np.nan,
            "probe_ok":False}
    p = Path(p)
    if p.suffix.lower() == ".wav":
        try:
            with wave.open(str(p), "rb") as wf:
                info.update(duration_sec=wf.getnframes()/wf.getframerate(), sample_rate=wf.getframerate(),
                            channels=wf.getnchannels(), audio_codec="pcm", probe_ok=True)
                return info
        except Exception: pass
    if FFPROBE is None: return info
    try:
        cmd = [FFPROBE, "-v", "error", "-show_entries",
               "format=duration:stream=codec_type,codec_name,sample_rate,channels,width,height,avg_frame_rate,r_frame_rate",
               "-of", "json", str(p)]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        if result.returncode != 0: return info
        data = json.loads(result.stdout)
        try: info["duration_sec"] = float(data.get("format", {}).get("duration", np.nan))
        except (ValueError,TypeError): pass
        for stream in data.get("streams", []):
            if stream.get("codec_type") == "audio" and info["audio_codec"] is None:
                info["audio_codec"] = stream.get("codec_name")
                info["channels"] = pd.to_numeric(stream.get("channels"),errors="coerce")
                info["sample_rate"] = pd.to_numeric(stream.get("sample_rate"),errors="coerce")
            if stream.get("codec_type") == "video" and info["video_codec"] is None:
                info["video_codec"] = stream.get("codec_name")
                info["width"] = pd.to_numeric(stream.get("width"),errors="coerce")
                info["height"] = pd.to_numeric(stream.get("height"),errors="coerce")
                fraction = stream.get("avg_frame_rate") or stream.get("r_frame_rate") or ""
                try:
                    n,d = fraction.split("/")
                    info["fps"] = float(n)/float(d) if float(d) else np.nan
                except (ValueError,ZeroDivisionError): pass
        info["probe_ok"] = np.isfinite(info["duration_sec"]) and info["duration_sec"]>0
    except (ValueError,subprocess.TimeoutExpired,OSError):
        pass
    return info

def media_section_file(ext_set, top):
    table = inventory[(inventory.section == top) & (inventory.extension.isin(ext_set))].copy()
    if len(table):
        table["subset"] = table.relative_path.map(lambda x: Path(x).parts[1] if len(Path(x).parts)>1 else "[root]")
    return table

audio_inventory = media_section_file(AUDIO_EXT, "01_audio")
audio_rows = []
for index, rec in enumerate(audio_inventory.itertuples(index=False),1):
    info = probe_media(rec.path)
    audio_rows.append({"relative_path":rec.relative_path, "subset":rec.subset,
                       "duration_sec":info["duration_sec"], "duration_h":info["duration_sec"]/3600,
                       "probe_ok":info["probe_ok"], "speaker_from_name":parse_speaker_id(rec.filename),
                       "sample_rate":info["sample_rate"], "audio_codec":info["audio_codec"]})
    if index%1000==0: print(f"  Аудіо перевірено: {index}/{len(audio_inventory)}")
audio = pd.DataFrame(audio_rows, columns=["relative_path", "subset", "duration_sec", "duration_h", "probe_ok",
                                         "speaker_from_name", "sample_rate", "audio_codec"])
audio.to_csv(OUT/"03_audio_durations.csv", index=False)
audio_summary = (audio.groupby("subset",dropna=False).agg(files=("relative_path","size"),
                hours=("duration_h","sum"), unreadable=("probe_ok",lambda x:(~x).sum()),
                speakers_by_name=("speaker_from_name","nunique")).reset_index())
display(audio_summary.round(3))
def audio_hours(subset):
    return float(audio.loc[audio.subset==subset,"duration_h"].sum())
OWN_H = audio_hours("own_processed")
EXTERNAL_H = audio_hours("external_selected")
SYNTH_H = audio_hours("synthetic")
CANONICAL_H = OWN_H + EXTERNAL_H + SYNTH_H
print(f"Власне оброблене: {OWN_H:.3f} год | зовнішнє: {EXTERNAL_H:.3f} год | синтетичне: {SYNTH_H:.3f} год")
print(f"Канонічний підсумок БЕЗ master і splits: {CANONICAL_H:.3f} год")
if (audio.subset=="own_processed").sum() == 0:
    print("УВАГА: own_processed порожній. own_master не зараховується автоматично як оброблений корпус.")


## 5. Відео: master, зовнішні фрагменти, маніпуляції

**Кількість файлів `cutted` не додаємо автоматично до кількості незалежних джерел:** вони можуть бути фрагментами одних і тих самих зовнішніх відео. Звіт розмежовує *фізичні файли*, *первинні master* та *похідні варіанти*.

In [ ]:
def guess_manipulation(path):
    p = "/" + str(path).lower().replace("\\","/") + "/"
    for name,tokens in [("fps_bitrate",["fps_bitrate"]),("audio_shift",["audio_shift"]),
                        ("tts_full",["tts_full"]),("local_audio_replace",["local_audio","audio_replace"]),
                        ("bitrate",["/bitrate/"]),("fps",["/fps/"]),
                        ("combined",["combined","combo"]),("frame_edit",["frame_drop","frame_edit"])]:
        if any(t in p for t in tokens):return name
    return "other_or_unknown"

video_inventory = media_section_file(VIDEO_EXT, "02_video")
video_rows = []
for index, rec in enumerate(video_inventory.itertuples(index=False),1):
    info = probe_media(rec.path)
    video_rows.append({"relative_path":rec.relative_path,"subset":rec.subset,
                       "duration_sec":info["duration_sec"], "fps":info["fps"],
                       "width":info["width"], "height":info["height"],
                       "probe_ok":info["probe_ok"], "codec":info["video_codec"],
                       "manipulation_type":guess_manipulation(rec.relative_path) if "manipulated" in rec.subset else None,
                       "speaker_from_name":parse_speaker_id(rec.filename),"filename":rec.filename})
    if index%100==0: print(f"  Відео перевірено: {index}/{len(video_inventory)}")
video = pd.DataFrame(video_rows, columns=["relative_path","subset","duration_sec","fps","width","height",
                                         "probe_ok","codec","manipulation_type","speaker_from_name","filename"])
video.to_csv(OUT/"04_video_probe.csv", index=False)
video_summary = video.groupby("subset",dropna=False).agg(files=("relative_path","size"),
                  total_minutes=("duration_sec",lambda s:s.sum()/60),
                  unreadable=("probe_ok",lambda x:(~x).sum()),avg_fps=("fps","mean")).reset_index()
display(video_summary.round(2))
manip = video[video.manipulation_type.notna()].groupby(["subset","manipulation_type"]).size().reset_index(name="files")
print("Типи маніпуляцій (визначені за шляхом; звіряти з manifest/receipts):")
display(manip)
manip.to_csv(OUT/"05_video_manipulations.csv", index=False)

mapping_path = DATA_ROOT/"02_video"/"own_manipulated"/"original_to_variant.csv"
video_mapping = read_table(mapping_path) if mapping_path.is_file() else None
if video_mapping is not None:
    print("original_to_variant.csv:", len(video_mapping), "рядків; поля:", list(video_mapping.columns))
    display(video_mapping.head(5))
else:
    print("original_to_variant.csv не знайдено або не прочитано: linkage буде CHECK.")


## 6. Текст: скільки насправді кейсів

Перевіряємо **лише** каталоги `03_text/own_cases` і `03_text/selected_cases`. Якщо для одного кейсу є і `.json`, і `.md`, файл не зараховується двічі. Якщо `case_id` відсутній, застосовується ідентифікатор із назви файла — це **оцінка**, а не підтверджена семантична унікальність.

Файли звітів, архівних версій і тестових шаблонів не враховуємо. Англійський переклад не є новим кейсом.

In [ ]:
TEXT_DIR = DATA_ROOT/"03_text"
CASE_SUBSETS = ["own_cases", "selected_cases"]
CASE_ID_FIELDS = ["case_id","question_id","claim_id","example_id","item_id","record_id"]
QUESTION_FIELDS = ["question_uk","question","claim","prompt","query","text"]
IGNORED_STEMS = ("report", "summary", "manifest", "schema", "readme", "template", "instructions", "license", "index")
PREF_EXT = {".json":0,".jsonl":1,".csv":2,".tsv":3,".parquet":4,".xlsx":5,".md":6,".txt":7}

def norm_text(v):
    if v is None or isinstance(v,float) and pd.isna(v):return ""
    return re.sub(r"\s+"," ",str(v)).strip().casefold()

def record_from_row(obj, fallback, subset, source):
    low = {str(k).lower():k for k in obj.keys()}
    explicit = next((str(obj[low[k]]).strip() for k in CASE_ID_FIELDS
                     if k in low and pd.notna(obj[low[k]]) and str(obj[low[k]]).strip()), None)
    q = next((str(obj[low[k]]).strip() for k in QUESTION_FIELDS
              if k in low and pd.notna(obj[low[k]]) and str(obj[low[k]]).strip()), "")
    has_uk = "question_uk" in low and norm_text(obj[low["question_uk"]]) != ""
    has_en = "question_en" in low and norm_text(obj[low["question_en"]]) != ""
    return {"subset":subset,"source_file":source,"case_id":explicit or fallback,
            "explicit_id":bool(explicit),"has_question":bool(q),"has_uk_en":bool(has_uk and has_en),
            "question_normalized":norm_text(q)}

case_rows=[]; reading_issues=[]
for subset in CASE_SUBSETS:
    root=TEXT_DIR/subset
    if not root.is_dir():
        reading_issues.append(f"Папка не знайдена: {root}")
        continue
    fs = sorted((p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in PREF_EXT and
                not any(p.stem.lower().startswith(x) for x in IGNORED_STEMS)),
                key=lambda p:(p.parent.as_posix(),p.stem,PREF_EXT[p.suffix.lower()]))
    # Один stem може бути представлений і у JSON, і у Markdown.
    per_stem={}
    for p in fs:
        key=(p.parent.as_posix(), p.stem)
        if key not in per_stem:per_stem[key]=p
    for p in per_stem.values():
        source=relpath(p)
        if p.suffix.lower() in (".md",".txt"):
            content=p.read_text(encoding="utf-8",errors="replace")
            case_rows.append({"subset":subset,"source_file":source,"case_id":f"{subset}::{p.stem}",
                              "explicit_id":False,"has_question":len(content.strip())>0,
                              "has_uk_en":False,"question_normalized":norm_text(content[:600])})
            continue
        df=read_table(p)
        if df is None or df.empty:
            reading_issues.append(f"Не прочитано як таблицю кейсів: {source}")
            continue
        # Не кожен рядок JSON є кейсом: потрібні принаймні текстові поля або ID.
        low_cols={str(c).lower() for c in df.columns}
        if not low_cols.intersection(set(CASE_ID_FIELDS+QUESTION_FIELDS+["question_en"])):
            reading_issues.append(f"Немає полів кейсу: {source}")
            continue
        for i, rec in df.iterrows():
            fallback=f"{subset}::{p.stem}::{i}"
            case_rows.append(record_from_row(rec.to_dict(),fallback,subset,source))
text_cases=pd.DataFrame(case_rows,columns=["subset","source_file","case_id","explicit_id","has_question","has_uk_en","question_normalized"])
text_cases.to_csv(OUT/"06_text_case_candidates.csv",index=False)
text_unique=text_cases.drop_duplicates("case_id")
text_summary=(text_unique.groupby("subset").agg(unique_candidate_ids=("case_id","size"),
              explicit_ids=("explicit_id","sum"), bilingual_confirmed=("has_uk_en","sum"),
              files=("source_file","nunique")).reset_index()) if not text_unique.empty else pd.DataFrame()
print("Кандидатів на унікальні кейси:",len(text_unique),"| із явними ID:",int(text_unique.explicit_id.sum()) if len(text_unique) else 0)
display(text_summary)
print("Складні для автоматичного підрахунку файли:",len(reading_issues))
(OUT/"text_count_warnings.txt").write_text("\n".join(reading_issues),encoding="utf-8")
# Перевірка дослівних дублікатів запитань, тільки там, де доступний текст.
valid_text=text_cases[text_cases.question_normalized.str.len()>15]
text_duplicate_candidates=(valid_text[valid_text.duplicated("question_normalized",keep=False)]
                          .sort_values("question_normalized"))
text_duplicate_candidates.to_csv(OUT/"07_text_possible_duplicates.csv",index=False)
print("Рядків із можливим повтором тексту:",len(text_duplicate_candidates))


## 7. Split: точні копії, групи мовців, похідні відео

Спочатку перевіряємо **однакові медіафайли** в різних split за SHA-256. Хешування проводиться лише для файлів, розміри яких збігаються між різними split, щоб не витрачати зайвий час. Потім окремо переглядаємо можливі збіги `speaker_id` і метадані `master → variant`.

**Увага:** відсутність SHA-дублікатів **не доводить** відсутність витоку між оригіналом і його стиснутою/зміщеною версією.

In [ ]:
split_rows=[]
SPLIT_ROOT=DATA_ROOT/"05_splits"
SPLIT_NAMES={"development","validation","hidden_test","evaluation"}
for subset in SPLIT_NAMES:
    d=SPLIT_ROOT/subset
    if not d.is_dir():continue
    for p in d.rglob("*"):
        if p.is_file() and not p.is_symlink():
            split_rows.append({"split":subset,"relative_path":relpath(p),"filename":p.name,
                               "extension":p.suffix.lower(),"size_bytes":p.stat().st_size,
                               "speaker_from_name":parse_speaker_id(p.name)})
split_files=pd.DataFrame(split_rows,columns=["split","relative_path","filename","extension","size_bytes","speaker_from_name"])
split_files.to_csv(OUT/"08_split_files.csv",index=False)
if len(split_files):
    split_summary=split_files.groupby("split").agg(files=("relative_path","size"),
                          gib=("size_bytes",lambda x:x.sum()/1024**3),
                          speaker_ids_from_name=("speaker_from_name","nunique")).reset_index()
    display(split_summary.round(3))
else:
    print("Splits не знайдено")

media_splits=split_files[split_files.extension.isin(AUDIO_EXT|VIDEO_EXT)].copy()
size_split_counts=media_splits.groupby("size_bytes").split.nunique() if len(media_splits) else pd.Series(dtype=int)
candidate_sizes=set(size_split_counts[size_split_counts>1].index)
hash_candidates=media_splits[media_splits.size_bytes.isin(candidate_sizes)]
hash_rows=[]
if CHECK_EXACT_SPLIT_DUPLICATES:
    print(f"Хешуємо {len(hash_candidates)} кандидатів (із {len(media_splits)} медіафайлів у split)")
    for j,rec in enumerate(hash_candidates.itertuples(index=False),1):
        h=hashlib.sha256()
        try:
            with (DATA_ROOT/rec.relative_path).open("rb") as f:
                for block in iter(lambda:f.read(8*1024*1024),b""):
                    h.update(block)
            value=h.hexdigest()
        except OSError:value=None
        hash_rows.append({"split":rec.split,"relative_path":rec.relative_path,"sha256":value})
        if j%500==0:print(f"  SHA-256: {j}/{len(hash_candidates)}")
    hashes=pd.DataFrame(hash_rows,columns=["split","relative_path","sha256"])
    cross=hashes.groupby("sha256").split.nunique() if len(hashes) else pd.Series(dtype=int)
    bad_hashes=set(cross[cross>1].index)-{None}
    exact_leak=hashes[hashes.sha256.isin(bad_hashes)].sort_values(["sha256","split"])
    hash_failures=int(hashes.sha256.isna().sum())
else:
    exact_leak=pd.DataFrame(columns=["split","relative_path","sha256"])
    bad_hashes=set();hash_failures=0
exact_leak.to_csv(OUT/"09_exact_cross_split_leakage.csv",index=False)
print(f"Перехресних SHA-груп: {len(bad_hashes)} | помилок хешування: {hash_failures}")

speaker_guesses=split_files.dropna(subset=["speaker_from_name"])
sp_multi=speaker_guesses.groupby("speaker_from_name").split.nunique() if len(speaker_guesses) else pd.Series(dtype=int)
suspect_speakers=set(sp_multi[sp_multi>1].index)
speaker_leaks=speaker_guesses[speaker_guesses.speaker_from_name.isin(suspect_speakers)]
speaker_leaks.to_csv(OUT/"10_possible_speaker_leakage.csv",index=False)
print("Імовірні speaker ID у >1 split (за назвою):",len(suspect_speakers))

# Перевірка mapping (тільки якщо є посилання на конкретні файли).
map_leak_rows=[]; matched_mapping_pairs=0
if video_mapping is not None and len(video_mapping):
    low={str(c).lower():c for c in video_mapping.columns}
    orig=next((col for name,col in low.items() if any(t in name for t in ("original","source_file","master_file","parent_file"))),None)
    var=next((col for name,col in low.items() if any(t in name for t in ("variant","output_file","manipulated_file"))),None)
    if orig is not None and var is not None:
        lookup=defaultdict(set)
        for rec in split_files.itertuples(index=False):lookup[rec.filename].add(rec.split)
        for _,r in video_mapping.iterrows():
            original=Path(str(r.get(orig,""))).name
            variant=Path(str(r.get(var,""))).name
            a,b=lookup.get(original,set()),lookup.get(variant,set())
            if a and b:
                matched_mapping_pairs+=1
                if a != b:
                    map_leak_rows.append({"original":original,"variant":variant,
                                          "original_splits":" | ".join(sorted(a)),
                                          "variant_splits":" | ".join(sorted(b))})
mapping_leak=pd.DataFrame(map_leak_rows,columns=["original","variant","original_splits","variant_splits"])
mapping_leak.to_csv(OUT/"11_mapping_derivative_leakage.csv",index=False)
print(f"Mapping підтверджених пар у split: {matched_mapping_pairs}; конфліктів: {len(mapping_leak)}")


## 8. Метадані: згода, регіон, походження, ліцензія

Автоматично переглянемо наявність ключових полів, **але не відкриватимемо вміст персональних форм згоди**. Якщо файл згоди існує, це ще не означає, що всі мовці надали необхідні дозволи.

In [ ]:
critical_fields = ["speaker_id","person_id","speaker_region","text_regional_profile",
                   "source_dataset","source_record_id","license","consent_status","split",
                   "original_id","parent_id","case_id","language","sha256"]
meta_col_names=set()
for colstr in metadata_inventory.get("columns",pd.Series(dtype=str)).dropna():
    meta_col_names.update(x.strip().lower() for x in colstr.split(" | "))
field_table=pd.DataFrame([{"field":f,"found_in_any_table":f in meta_col_names} for f in critical_fields])
field_table.to_csv(OUT/"12_metadata_fields.csv",index=False)
display(field_table)
consent_dir=DATA_ROOT/"04_docs"/"consent"
consent_count=len([p for p in consent_dir.rglob("*") if p.is_file()]) if consent_dir.is_dir() else 0
print("Файлів у 04_docs/consent:",consent_count, "(НЕ підтверджує підписання всіма учасниками)")
print("Ліцензії зовнішніх джерел потребують перевірки по кожній версії даних та кожному файлу.")


## 9. Підсумкова таблиця KPI

`PASS` — виміряний кількісний показник досягнуто; `FAIL` — достовірно нижче цілі; `CHECK` — інформації недостатньо для остаточного приймання. **Попередній підрахунок файлів не дорівнює сертифікованій якості даних.**

In [ ]:
def status_ge(actual,required,missing=False):
    if missing or pd.isna(actual):return "CHECK"
    return "PASS" if actual>=required else "FAIL"

def subset_video_count(name):return int((video.subset==name).sum()) if len(video) else 0

def subset_missing_audio(name):return int((~audio.loc[audio.subset==name,"probe_ok"]).sum()) if len(audio) else 0

own_master_video=subset_video_count("own_master")
external_master_video=subset_video_count("external_master")
cut_videos=subset_video_count("cutted")
manip_videos=subset_video_count("own_manipulated")+subset_video_count("manipulated")
own_cases=int((text_unique.subset=="own_cases").sum()) if len(text_unique) else 0
rows_kpi=[]
def add(section,metric,target,actual,status,note=""):
    rows_kpi.append({"component":section,"metric":metric,"target":target,"actual":actual,"status":status,"note":note})

missing_canonical=sum(subset_missing_audio(s) for s in ("own_processed","external_selected","synthetic"))
add("AUDIO","Канонічний обсяг",">= 10 год",round(CANONICAL_H,3),status_ge(CANONICAL_H,10,missing_canonical>0),"own_master та splits не додаються")
add("AUDIO","Власні очищені записи",">= 3 год",round(OWN_H,3),status_ge(OWN_H,3,subset_missing_audio("own_processed")>0))
add("AUDIO","Реальне мовлення",">= 5 год",round(OWN_H+EXTERNAL_H,3),status_ge(OWN_H+EXTERNAL_H,5,missing_canonical>0),"Папка external_selected умовно вважається real — треба звірити labels")
add("AUDIO","Синтетичне мовлення",">= 5 год",round(SYNTH_H,3),status_ge(SYNTH_H,5,subset_missing_audio("synthetic")>0))
add("AUDIO","Мовці 3 регіональних груп","3 групи","не перевірено","CHECK","Читання регіонального тексту не означає походження мовця")
add("VIDEO","Власні master-фрагменти",">= 24",own_master_video,status_ge(own_master_video,24))
add("VIDEO","Зовнішні master-фрагменти",">= 24",external_master_video,status_ge(external_master_video,24))
add("VIDEO","Незалежні українські первинні фрагменти",">= 80",f"{own_master_video} власних + {external_master_video} зовн. + {cut_videos} cutted","CHECK","Cutted можуть походити з тих самих master")
add("VIDEO","Маніпульовані медіаваріанти",">= 80",manip_videos,status_ge(manip_videos,80),"Перевірка різноманітності типів окремо")
add("TEXT","Унікальні семантичні двомовні кейси",">= 300",len(text_unique),"CHECK","Автоматично підтверджено тільки кількість кандидатів на case_id")
add("TEXT","Нові власні кейси",">= 90",own_cases,"CHECK" if own_cases>=90 else "FAIL","Без експертної верифікації унікальності")
if CHECK_EXACT_SPLIT_DUPLICATES:
    exact_status="CHECK" if hash_failures>0 else ("PASS" if not bad_hashes else "FAIL")
else:exact_status="CHECK"
add("SPLITS","Однакові медіафайли між split","0",len(bad_hashes),exact_status,"SHA-256; не виявляє перетворені копії")
add("SPLITS","Мовці між split","0",len(suspect_speakers),"CHECK","Евристика з filename; потрібні перевірені speaker_id")
add("SPLITS","Master→variant між split","0",len(mapping_leak),"FAIL" if len(mapping_leak) else "CHECK",f"Зіставлено {matched_mapping_pairs} пар із mapping")
add("LEGAL","Права й згоди для публічного релізу","100%","не перевірено","CHECK","Ручна перевірка ліцензій, дозволів і consent")
add("QUALITY","Необроблені помилки читання аудіо","0",int((~audio.probe_ok).sum()) if len(audio) else 0,
    "PASS" if len(audio) and audio.probe_ok.all() else "CHECK")
add("QUALITY","Необроблені помилки читання відео","0",int((~video.probe_ok).sum()) if len(video) else 0,
    "PASS" if len(video) and video.probe_ok.all() else "CHECK")
kpi=pd.DataFrame(rows_kpi)
kpi.to_csv(OUT/"13_FINAL_KPI_ACCEPTANCE.csv",index=False)
display(kpi)


## 10. Графіки: що вже є, а чого бракує

Графіки використовують лише кількості, визначені вище; вони не замінюють перевірки метаданих.

In [ ]:
try:
    import matplotlib.pyplot as plt
    audio_names=["Власне real","Зовнішнє real*","Synthetic"]
    audio_values=[OWN_H,EXTERNAL_H,SYNTH_H]
    fig,ax=plt.subplots(figsize=(8,3.5))
    bars=ax.bar(audio_names,audio_values)
    ax.axhline(3.0,linestyle="--",linewidth=1,label="Мінімум власного: 3 год")
    for bar,v in zip(bars,audio_values):ax.text(bar.get_x()+bar.get_width()/2,v,f"{v:.2f}",ha="center",va="bottom")
    ax.set_ylabel("Години")
    ax.set_title("AUA: канонічний обсяг аудіо за категорією")
    ax.legend(loc="upper right")
    fig.tight_layout();fig.savefig(OUT/"audio_chart.png",dpi=145);plt.show()
    fig,ax=plt.subplots(figsize=(8,3.5))
    cats=["Власні master","Зовнішні master","Cutted","Маніпуляції"]
    vals=[own_master_video,external_master_video,cut_videos,manip_videos]
    bars=ax.bar(cats,vals)
    for b,v in zip(bars,vals):ax.text(b.get_x()+b.get_width()/2,v,str(v),ha="center",va="bottom")
    ax.set_ylabel("Кількість медіафайлів")
    ax.set_title("AUA: категорії відео (не сумувати як незалежні джерела)")
    fig.tight_layout();fig.savefig(OUT/"video_chart.png",dpi=145);plt.show()
except ImportError:
    print("Для графіків встановіть matplotlib. CSV-звіти вже сформовані.")


## 11. Підсумок та файли звіту

Після виконання завантажте в чат **`13_FINAL_KPI_ACCEPTANCE.csv`** та **`AUA_AUDIT_SUMMARY.md`**. Для розбору витоків додайте `09_exact_cross_split_leakage.csv`, `10_possible_speaker_leakage.csv` і `11_mapping_derivative_leakage.csv`, якщо вони не порожні.

In [ ]:
summary={
    "dataset_root":str(DATA_ROOT),"audit_output":str(OUT),"file_count":int(len(inventory)),
    "bytes_total":int(inventory.size_bytes.sum()),
    "audio":{"own_processed_h":round(OWN_H,4),"external_selected_h":round(EXTERNAL_H,4),
             "synthetic_h":round(SYNTH_H,4),"canonical_h":round(CANONICAL_H,4),
             "unreadable":int((~audio.probe_ok).sum())},
    "video":{"own_master_files":own_master_video,"external_master_files":external_master_video,
             "cutted_files":cut_videos,"manipulated_media_files":manip_videos,
             "unreadable":int((~video.probe_ok).sum())},
    "text":{"candidate_unique_case_ids":int(len(text_unique)),"own_candidates":own_cases,
            "explicit_ids":int(text_unique.explicit_id.sum()) if len(text_unique) else 0,
            "possible_text_duplicate_rows":int(len(text_duplicate_candidates))},
    "splits":{"exact_duplicate_hash_groups":int(len(bad_hashes)),
              "possible_speaker_ids_in_many_splits":int(len(suspect_speakers)),
              "mapping_pairs_linked":int(matched_mapping_pairs),
              "mapping_cross_split_conflicts":int(len(mapping_leak)),
              "hashing_performed":bool(CHECK_EXACT_SPLIT_DUPLICATES)},
    "kpi_statuses":kpi.status.value_counts().to_dict()
}
(OUT/"14_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")

lines=["# AUA v1 — висновок автоматичного аудиту","",
       f"Шлях до набору: `{DATA_ROOT}`",f"Файлів: **{len(inventory):,}**; обсяг: **{inventory.size_bytes.sum()/1024**3:.3f} GiB**.",
       "", "## Підсумкові показники", "",
       "| Категорія | Показник | Ціль | Факт | Статус |", "|---|---|---:|---:|---|"]
for r in kpi.itertuples(index=False):
    lines.append(f"| {r.component} | {r.metric.replace('|','/')} | {r.target} | {r.actual} | {r.status} |")
lines.extend(["", "## Обов’язкові застереження", "",
              "- Це аудит файлів, а не лабораторне або юридичне приймання набору.",
              "- Унікальний filename/case_id не гарантує унікальності особи, події чи змісту.",
              "- `cutted` не підсумовується з master без аналізу походження.",
              "- Перевірка SHA-256 не виявляє різнокодекові, стиснуті або змінені похідні.",
              "- Наявність каталогів зі згодами не доводить згоди всіх учасників.",
              "- Для захищеного hidden test має бути окремий контроль доступу і походження даних.",
              "", "## Які файли відкрити", "",
              "`13_FINAL_KPI_ACCEPTANCE.csv`, `03_audio_durations.csv`, `04_video_probe.csv`,",
              "`06_text_case_candidates.csv`, `09_exact_cross_split_leakage.csv`,",
              "`10_possible_speaker_leakage.csv`, `11_mapping_derivative_leakage.csv`." ])
summary_file=OUT/"AUA_AUDIT_SUMMARY.md"
summary_file.write_text("\n".join(lines),encoding="utf-8")
print("ГОТОВО. Збережено звіти:")
print(summary_file)
print(OUT/"13_FINAL_KPI_ACCEPTANCE.csv")
print(OUT/"14_summary.json")
print("Статуси:",summary["kpi_statuses"])
